# Global AI Adoption in Education: Exploratory Data Analysis (EDA)
**Capstone Project:** Data Analytics with Tableau  
**Author:** Portfolio Evaluation  
**Dataset:** `data/raw/Global AI in Education.csv` (1,360 rows, 16 features)

This notebook executes a rigorous, factual exploratory data analysis covering all 16 required analytical dimensions without fabricated statistics or assumptions.


## 1. Dataset Overview & Environment Setup
We load the raw CSV dataset into a pandas DataFrame. We explicitly set `keep_default_na=False` so that the legitimate category `"None"` in `government_ai_policy` is correctly preserved as a categorical string rather than erroneously converted to NaN.


In [ ]:
import pandas as pd
import numpy as np

# Load dataset with strict string preservation for policy field
raw_path = '../data/raw/Global AI in Education.csv'
df = pd.read_csv(raw_path, keep_default_na=False)

print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
print("\nColumn Data Types:")
print(df.dtypes)
df.head(10)


## 2. Missing-Value Analysis
We inspect each column for null, NaN, or empty values.


In [ ]:
missing_counts = df.isnull().sum()
empty_strings = (df == '').sum()
na_report = pd.DataFrame({
    'Missing (NaN)': missing_counts,
    'Empty Strings': empty_strings,
    'Data Type': df.dtypes
})
print("Missing Value Audit:")
print(na_report)
assert df.isnull().sum().sum() == 0, "No missing values present."
print("\nAudit Result: 0 missing or corrupted values detected.")


## 3. Duplicate Analysis
We verify whether any identical records exist across all 16 features.


In [ ]:
duplicates = df.duplicated().sum()
print(f"Duplicate Rows Count: {duplicates}")
assert duplicates == 0, "Dataset contains no duplicate observations."
print("Integrity Result: Full record uniqueness confirmed across all 1,360 monthly country observations.")


## 4. Descriptive Statistics
Summary statistics for all quantitative features including student usage, teacher usage, school adoption, daily usage, infrastructure metrics, and demographic gaps.


In [ ]:
numeric_cols = [
    'student_ai_usage_pct', 'teacher_ai_usage_pct', 'schools_ai_adoption_pct',
    'avg_daily_ai_usage_min', 'internet_penetration_pct', 'education_index',
    'urban_ai_usage_pct', 'rural_ai_usage_pct', 'gender_gap_ai_usage_pct'
]
desc_stats = df[numeric_cols].describe().round(2).T
desc_stats['range'] = desc_stats['max'] - desc_stats['min']
desc_stats[['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max', 'range']]


## 5. Country Analysis
Evaluating AI adoption across the 10 sovereign nations represented in the dataset. Each country has exactly 136 monthly observations spanning 2015 to 2026.


In [ ]:
country_summary = df.groupby('country').agg(
    record_count=('year', 'count'),
    region=('region', 'first'),
    avg_student_ai=('student_ai_usage_pct', 'mean'),
    avg_teacher_ai=('teacher_ai_usage_pct', 'mean'),
    avg_school_ai=('schools_ai_adoption_pct', 'mean'),
    avg_daily_min=('avg_daily_ai_usage_min', 'mean'),
    internet_pen=('internet_penetration_pct', 'first'),
    edu_index=('education_index', 'first'),
    avg_gender_gap=('gender_gap_ai_usage_pct', 'mean')
).round(2).sort_values(by='avg_student_ai', ascending=False)

country_summary


## 6. Regional Analysis
Aggregating the 10 countries into their 6 respective geographic regions: Africa, Asia, Europe, North America, Oceania, and South America.


In [ ]:
regional_summary = df.groupby('region').agg(
    total_records=('year', 'count'),
    num_countries=('country', 'nunique'),
    avg_student_ai=('student_ai_usage_pct', 'mean'),
    avg_teacher_ai=('teacher_ai_usage_pct', 'mean'),
    avg_school_ai=('schools_ai_adoption_pct', 'mean'),
    avg_internet=('internet_penetration_pct', 'mean'),
    avg_edu_index=('education_index', 'mean'),
    avg_gender_gap=('gender_gap_ai_usage_pct', 'mean')
).round(2).sort_values(by='avg_student_ai', ascending=False)

regional_summary


## 7. Student vs Teacher AI Adoption Comparison
Evaluating the relationship and gap between grassroots student adoption and educator/teacher adoption.


In [ ]:
student_teacher_df = df.copy()
student_teacher_df['student_teacher_diff'] = student_teacher_df['student_ai_usage_pct'] - student_teacher_df['teacher_ai_usage_pct']

print(f"Overall Mean Student Adoption: {student_teacher_df['student_ai_usage_pct'].mean():.2f}%")
print(f"Overall Mean Teacher Adoption: {student_teacher_df['teacher_ai_usage_pct'].mean():.2f}%")
print(f"Overall Mean Adoption Difference (Student - Teacher): {student_teacher_df['student_teacher_diff'].mean():.2f}%")

st_by_year = student_teacher_df.groupby('year')[['student_ai_usage_pct', 'teacher_ai_usage_pct', 'student_teacher_diff']].mean().round(2)
st_by_year


## 8. School-Level AI Adoption Analysis
Institutional school adoption measures the percentage of academic institutions that formally adopt AI.


In [ ]:
school_comparison = df.groupby('year').agg(
    avg_student_ai=('student_ai_usage_pct', 'mean'),
    avg_teacher_ai=('teacher_ai_usage_pct', 'mean'),
    avg_school_ai=('schools_ai_adoption_pct', 'mean')
).round(2)

print("Institutional School Adoption vs User Adoption:")
print(school_comparison)


## 9. AI Tool Distribution Analysis
Examining the occurrence frequency of the 4 primary educational AI tools (ChatGPT, Google Gemini, Khanmigo, Microsoft Copilot) across global observations and geographic regions.


In [ ]:
tool_dist = df['top_ai_tool'].value_counts()
tool_pct = (df['top_ai_tool'].value_counts(normalize=True) * 100).round(2)

tool_summary = pd.DataFrame({
    'Frequency (Months As Top Tool)': tool_dist,
    'Share (%)': tool_pct
})
print("Global Tool Distribution:")
print(tool_summary)

print("\nTop AI Tool Distribution by Region:")
tool_by_region = pd.crosstab(df['region'], df['top_ai_tool'], margins=True)
tool_by_region


## 10. Internet Penetration Analysis
Analyzing how national internet penetration correlates with student, teacher, and school adoption rates.


In [ ]:
inet_analysis = df.groupby('country').agg(
    internet_penetration=('internet_penetration_pct', 'first'),
    avg_student_ai=('student_ai_usage_pct', 'mean'),
    avg_teacher_ai=('teacher_ai_usage_pct', 'mean'),
    avg_school_ai=('schools_ai_adoption_pct', 'mean'),
    avg_daily_min=('avg_daily_ai_usage_min', 'mean'),
    avg_rural_ai=('rural_ai_usage_pct', 'mean')
).round(2).sort_values(by='internet_penetration', ascending=False)

print("Country Level Internet Penetration vs Educational Metrics:")
print(inet_analysis)

# Correlation between internet penetration and AI usage
corr_inet_student = df['internet_penetration_pct'].corr(df['student_ai_usage_pct'])
print(f"\nCorrelation between Internet Penetration and Student AI Usage: {corr_inet_student:.4f}")


## 11. Education Index Analysis
Assessing how the United Nations Education Index score relates to AI adoption and engagement intensity.


In [ ]:
edu_analysis = df.groupby('country').agg(
    education_index=('education_index', 'first'),
    avg_student_ai=('student_ai_usage_pct', 'mean'),
    avg_teacher_ai=('teacher_ai_usage_pct', 'mean'),
    avg_school_ai=('schools_ai_adoption_pct', 'mean'),
    avg_daily_min=('avg_daily_ai_usage_min', 'mean')
).round(2).sort_values(by='education_index', ascending=False)

print("Education Index vs AI Adoption:")
print(edu_analysis)

corr_edu_student = df['education_index'].corr(df['student_ai_usage_pct'])
print(f"\nCorrelation between Education Index and Student AI Usage: {corr_edu_student:.4f}")


## 12. Government AI Policy Analysis
Investigating adoption rates grouped by national policy status: `Active`, `Draft`, and `None`.


In [ ]:
policy_summary = df.groupby('government_ai_policy').agg(
    record_count=('year', 'count'),
    share_pct=('year', lambda x: round(len(x) / len(df) * 100, 2)),
    avg_student_ai=('student_ai_usage_pct', 'mean'),
    avg_teacher_ai=('teacher_ai_usage_pct', 'mean'),
    avg_school_ai=('schools_ai_adoption_pct', 'mean'),
    avg_daily_min=('avg_daily_ai_usage_min', 'mean')
).round(2).loc[['Active', 'Draft', 'None']]

print("Adoption Metrics by Government AI Policy Status:")
print(policy_summary)


## 13. Urban vs Rural Adoption Gap Analysis
Analyzing geographic equity by measuring urban and rural AI adoption trajectories over time.


In [ ]:
urban_rural_df = df.copy()
urban_rural_df['divide_gap'] = urban_rural_df['urban_ai_usage_pct'] - urban_rural_df['rural_ai_usage_pct']

print(f"Overall Mean Urban AI Usage: {urban_rural_df['urban_ai_usage_pct'].mean():.2f}%")
print(f"Overall Mean Rural AI Usage: {urban_rural_df['rural_ai_usage_pct'].mean():.2f}%")
print(f"Overall Mean Urban-Rural Divide Gap: {urban_rural_df['divide_gap'].mean():.2f}%")

divide_by_year = urban_rural_df.groupby('year').agg(
    urban_avg=('urban_ai_usage_pct', 'mean'),
    rural_avg=('rural_ai_usage_pct', 'mean'),
    gap_avg=('divide_gap', 'mean')
).round(2)

print("\nUrban vs Rural AI Usage Evolution by Year:")
print(divide_by_year)


## 14. AI in Curriculum Analysis
Comparing educational environments where AI is formally integrated into curriculum (`Yes`) versus those where it is not (`No`).


In [ ]:
curriculum_summary = df.groupby('ai_in_curriculum').agg(
    observations=('year', 'count'),
    share_pct=('year', lambda x: round(len(x) / len(df) * 100, 2)),
    avg_student_ai=('student_ai_usage_pct', 'mean'),
    avg_teacher_ai=('teacher_ai_usage_pct', 'mean'),
    avg_school_ai=('schools_ai_adoption_pct', 'mean'),
    avg_urban_ai=('urban_ai_usage_pct', 'mean'),
    avg_rural_ai=('rural_ai_usage_pct', 'mean')
).round(2).loc[['Yes', 'No']]

print("AI in Curriculum Comparison:")
print(curriculum_summary)


## 15. Gender Gap Analysis
Examining the gender gap in AI usage across countries, regions, and over time.


In [ ]:
gender_by_region = df.groupby('region').agg(
    avg_gender_gap=('gender_gap_ai_usage_pct', 'mean'),
    min_gender_gap=('gender_gap_ai_usage_pct', 'min'),
    max_gender_gap=('gender_gap_ai_usage_pct', 'max')
).round(2).sort_values(by='avg_gender_gap', ascending=False)

print("Gender Gap in AI Usage by Region:")
print(gender_by_region)

gender_by_year = df.groupby('year')['gender_gap_ai_usage_pct'].agg(
    ['mean', 'min', 'max']
).round(2)
print("\nGender Gap Stability Across Years:")
print(gender_by_year)


## 16. Temporal Trends (Year & Month Trajectories)
Synthesizing the multi-year progression from early adoption in 2015 to widespread integration in 2026.


In [ ]:
timeline = df.groupby('year').agg(
    records=('country', 'count'),
    student_ai=('student_ai_usage_pct', 'mean'),
    teacher_ai=('teacher_ai_usage_pct', 'mean'),
    school_ai=('schools_ai_adoption_pct', 'mean'),
    daily_min=('avg_daily_ai_usage_min', 'mean'),
    urban_ai=('urban_ai_usage_pct', 'mean'),
    rural_ai=('rural_ai_usage_pct', 'mean')
).round(2)

print("12-Year Temporal Trajectory Summary:")
print(timeline)
print("\nKey Takeaway: Student usage escalated from 4.98% in 2015 to 60.75% in 2026, with teacher and school adoption following an almost identical trajectory.")
